In [ ]:
# Weights notebook (no submission): download zai-org/GLM-4-9B-0414 (MIT) from Hugging Face and save only the first
# KEEP decoder layers (+ embeddings) in fp16, so a submission notebook can attach this output offline.
# hidden_states[L] for L < KEEP is identical to the full model's (truncation verified exact on qwen3/llama/gemma2).
HF_ID, KEEP, OUT = 'zai-org/GLM-4-9B-0414', 25, 'glm4_9b_trunc25'

import json
import os
import shutil
import time

import torch
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoModel, AutoTokenizer

t0 = time.time()
src = snapshot_download(HF_ID, cache_dir='/tmp/hf', allow_patterns=['*.json', '*.safetensors', '*.model', '*.txt', '*.py',
                                                                   '*.tiktoken', 'tokenizer*'])
print('downloaded', src, f'{time.time() - t0:.0f}s', flush=True)
cfg = AutoConfig.from_pretrained(src)
print('model_type', cfg.model_type, 'hidden', cfg.hidden_size, 'layers', cfg.num_hidden_layers, flush=True)
cfg.num_hidden_layers = KEEP
if getattr(cfg, 'layer_types', None):
    cfg.layer_types = cfg.layer_types[:KEEP]
model = AutoModel.from_pretrained(src, config=cfg, dtype=torch.float16, low_cpu_mem_usage=True)
model.save_pretrained(OUT, safe_serialization=True)
AutoTokenizer.from_pretrained(src).save_pretrained(OUT)
json.dump({'source': HF_ID, 'keep_layers': KEEP, 'orig_layers': AutoConfig.from_pretrained(src).num_hidden_layers},
          open(os.path.join(OUT, 'trunc_info.json'), 'w'))
shutil.rmtree('/tmp/hf', ignore_errors=True)
print('saved', OUT, os.listdir(OUT), f'{time.time() - t0:.0f}s', flush=True)
import pandas as pd
pd.DataFrame({'id': [], 'model': []}).to_parquet('submission.parquet')   # placeholder
